# PdM-Lite — CWRU Bearing Fault Classifier

Trains a small 1D-CNN on the **CWRU Bearing Dataset** to classify four conditions:
`normal` / `inner_race` / `ball` / `outer_race`.
Exports the trained model to **ONNX** for CPU inference in the FastAPI service.

**Runtime:** GPU recommended (Runtime → Change runtime type → T4 GPU), but CPU works too.

**Outputs produced:**
- `model.onnx` — download and place in `app/artifacts/`
- `model_config.json` — class names + input shape (used by `app/model.py`)
- `training_curves.png` / `confusion_matrix.png` — for the README

## 0 · Install dependencies

In [ ]:
!pip install -q scipy onnx onnxscript onnxruntime scikit-learn seaborn

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

import onnx
import onnxruntime as ort

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1 · Download CWRU Data

Drive-End bearing data at **12 kHz**, fault diameter **0.007 in**, all four load conditions (0–3 HP).

| Label | Class | Files (file numbers) |
|-------|-------|----------------------|
| 0 | normal | 97, 98, 99, 100 |
| 1 | inner_race | 105, 106, 107, 108 |
| 2 | ball | 118, 119, 120, 121 |
| 3 | outer_race | 130, 131, 132, 133 |

> If a download fails, manually grab the `.mat` files from  
> https://engineering.case.edu/bearingdatacenter/download-data-file  
> and upload them to `cwru_data/` in this session.

In [ ]:
import requests

DATA_DIR = Path('cwru_data')
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = 'https://engineering.case.edu/sites/default/files/'

FILE_GROUPS = {
    'normal':     [97, 98, 99, 100],
    'inner_race': [105, 106, 107, 108],
    'ball':       [118, 119, 120, 121],
    'outer_race': [130, 131, 132, 133],
}

CLASS_NAMES = list(FILE_GROUPS.keys())  # order defines label 0-3

for cls_name, file_nums in FILE_GROUPS.items():
    for num in file_nums:
        dest = DATA_DIR / f'{num}.mat'
        if dest.exists():
            print(f'  {dest.name} already downloaded')
            continue
        url = BASE_URL + f'{num}.mat'
        print(f'Downloading {url} ...')
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        print(f'  saved {dest.name} ({len(resp.content) // 1024} KB)')

print('\nAll files present.')

## 2 · Load Signals & Create Windows

Each `.mat` file contains a long raw vibration time-series.  
We slice it into **1024-sample windows with 50% overlap** to create training examples.

In [ ]:
WINDOW_SIZE = 1024
STRIDE = 512  # 50% overlap


def extract_de_signal(mat_path):
    """Extract the Drive-End accelerometer channel from a CWRU .mat file."""
    mat = scipy.io.loadmat(str(mat_path))
    for key, val in mat.items():
        if 'DE_time' in key:
            return val.squeeze().astype(np.float32)
    raise KeyError(f'No DE_time variable found in {mat_path}')


def sliding_windows(signal, size=WINDOW_SIZE, stride=STRIDE):
    n = (len(signal) - size) // stride + 1
    idx = np.arange(n)[:, None] * stride + np.arange(size)[None, :]
    return signal[idx]


X_parts, y_parts = [], []

for label, (cls_name, file_nums) in enumerate(FILE_GROUPS.items()):
    for num in file_nums:
        signal = extract_de_signal(DATA_DIR / f'{num}.mat')
        windows = sliding_windows(signal)
        X_parts.append(windows)
        y_parts.append(np.full(len(windows), label, dtype=np.int64))
        print(f'  {cls_name:12s} file {num}: {len(signal):>7,} samples → {len(windows):>5,} windows')

X = np.concatenate(X_parts)        # (N, 1024)
y = np.concatenate(y_parts)        # (N,)

print(f'\nDataset shape : {X.shape}')
print(f'Class counts  : {dict(zip(CLASS_NAMES, np.bincount(y)))}')

## 3 · Normalize & Split

In [ ]:
# Per-window z-score normalisation
mu = X.mean(axis=1, keepdims=True)
sigma = X.std(axis=1, keepdims=True) + 1e-8
X = (X - mu) / sigma

# Add channel dimension: (N, 1, 1024)
X = X[:, np.newaxis, :]

# Stratified 70 / 15 / 15 split
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)

print(f'Train {len(y_train):>6,} | Val {len(y_val):>6,} | Test {len(y_test):>6,}')

## 4 · PyTorch Dataset & DataLoaders

In [ ]:
class BearingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


BATCH_SIZE = 64

train_loader = DataLoader(BearingDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(BearingDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(BearingDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

## 5 · Model Architecture

A small three-block 1D-CNN with BatchNorm.  Input shape: `(batch, 1, 1024)`.  
~50 k parameters — fast on CPU at inference time.

```
Input  (B, 1, 1024)
  Conv1d(1→32, k=64, s=2)  → (B, 32, 481)
  BN + ReLU + MaxPool(2)   → (B, 32, 240)
  Conv1d(32→64, k=32, s=2) → (B, 64, 105)
  BN + ReLU + MaxPool(2)   → (B, 64, 52)
  Conv1d(64→128, k=16, s=2)→ (B, 128, 19)
  BN + ReLU + AvgPool→1    → (B, 128, 1)
  Flatten → Linear(128, 4) → (B, 4) logits
```

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,  32,  kernel_size=64, stride=2), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64,  kernel_size=32, stride=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=16, stride=2), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = CNN1D(n_classes=4).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# Shape sanity-check
with torch.no_grad():
    dummy = torch.zeros(2, 1, 1024, device=DEVICE)
    out = model(dummy)
print(f'Output shape: {out.shape}  (expected (2, 4))')

## 6 · Training

In [ ]:
EPOCHS   = 40
PATIENCE = 6

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
best_state    = None
patience_cnt  = 0


def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n


for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'train loss {tr_loss:.4f} acc {tr_acc:.4f}  '
          f'val loss {vl_loss:.4f} acc {vl_acc:.4f}')

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt  = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

model.load_state_dict(best_state)
print('Best checkpoint restored.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='train'); ax1.plot(history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_acc'], label='train'); ax2.plot(history['val_acc'], label='val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()
print('Saved training_curves.png')

## 7 · Test-Set Evaluation

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

report = classification_report(all_true, all_preds, target_names=CLASS_NAMES, output_dict=True)
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES))

# ── clear summary for README ──────────────────────────────────────────────────
print("=" * 50)
print(f"Overall test accuracy : {report['accuracy']*100:.2f}%")
for cls in CLASS_NAMES:
    print(f"  {cls:<12} F1 = {report[cls]['f1-score']:.4f}")
print("=" * 50)

In [ ]:
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()
print('Saved confusion_matrix.png')

## 8 · Export to ONNX

Input name: `signal` — shape `(batch, 1, 1024)`, dtype `float32`  
Output name: `logits` — shape `(batch, 4)`, dtype `float32`  
Batch dimension is dynamic so the API can run single-sample or batched inference.

In [ ]:
model.eval().cpu()

dummy_input = torch.zeros(1, 1, 1024)

torch.onnx.export(
    model,
    dummy_input,
    'model.onnx',
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=['signal'],
    output_names=['logits'],
    dynamic_axes={'signal': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
)
print(f'Exported model.onnx ({Path("model.onnx").stat().st_size // 1024} KB)')

In [ ]:
# Structural check
onnx_model = onnx.load('model.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX structural check passed.')

# Runtime check — compare PyTorch and ONNX outputs on the same input
sess = ort.InferenceSession('model.onnx', providers=['CPUExecutionProvider'])
sample_np = X_test[:8].astype(np.float32)

ort_logits = sess.run(['logits'], {'signal': sample_np})[0]

with torch.no_grad():
    pt_logits = model(torch.from_numpy(sample_np)).numpy()

max_diff = np.abs(ort_logits - pt_logits).max()
print(f'Max abs diff (PyTorch vs ONNX): {max_diff:.2e}  (should be < 1e-4)')
assert max_diff < 1e-3, 'ONNX output diverges from PyTorch — check the export'
print('ONNX runtime check passed.')

## 9 · Save Config & Download Files

Download `model.onnx` and `model_config.json`, then place them in `app/artifacts/` in the repo.

In [ ]:
config = {
    'class_names': CLASS_NAMES,
    'input_name':  'signal',
    'output_name': 'logits',
    'window_size': WINDOW_SIZE,
    'n_classes':   len(CLASS_NAMES),
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved model_config.json')
print(json.dumps(config, indent=2))

In [ ]:
from google.colab import files

for fname in ['model.onnx', 'model_config.json', 'training_curves.png', 'confusion_matrix.png']:
    files.download(fname)

print('Download complete. Place model.onnx and model_config.json in app/artifacts/')